## RoBERTa

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from sklearn.metrics import f1_score, accuracy_score

# ── Hyper-parameters ──────────────────────────────────────────────────────────
MODEL_NAME  = "roberta-large"  
NUM_CLASSES = 7
MAX_LEN     = 128             
EPOCHS      = 5                
BATCH_SIZE  = 32               
LR          = 2e-5             
SEED        = 42
# ─────────────────────────────────────────────────────────────────────────────

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    macro_f1 = f1_score(labels, predictions, average="macro", zero_division=0)
    acc = accuracy_score(labels, predictions)
    
    return {"macro_f1": macro_f1, "accuracy": acc}

def main():
    print(f"Loading {MODEL_NAME} tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_df = pd.read_csv("data/train.csv")
    valid_df = pd.read_csv("data/valid.csv")
    test_df  = pd.read_csv("data/test_no_label.csv")
    train_ds = Dataset.from_pandas(train_df[["text", "label"]])
    valid_ds = Dataset.from_pandas(valid_df[["text", "label"]])
    test_ds  = Dataset.from_pandas(test_df[["id", "text"]])

    def tokenize_function(examples):
        return tokenizer(
            examples["text"], 
            truncation=True, 
            max_length=MAX_LEN
        )

    print("Tokenizing datasets...")
    train_tokenized = train_ds.map(tokenize_function, batched=True)
    valid_tokenized = valid_ds.map(tokenize_function, batched=True)
    test_tokenized  = test_ds.map(tokenize_function, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    print(f"Loading {MODEL_NAME} model for sequence classification...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=NUM_CLASSES
    )

    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",           
        save_strategy="epoch",
        save_total_limit=2,              
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE*2,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        fp16=True,                        
        load_best_model_at_end=True,     
        metric_for_best_model="macro_f1", 
        seed=SEED,
        logging_steps=50,
        report_to="none"                
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=valid_tokenized,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print("Starting training...")
    trainer.train()

    print("\nGenerating predictions on test set...")
    predictions = trainer.predict(test_tokenized)
    preds = np.argmax(predictions.predictions, axis=-1)

    out = pd.DataFrame({"id": test_df["id"], "label": preds})
    out.to_csv("roberta_pred.csv", index=False)
    print("Saved roberta_pred.csv")

if __name__ == "__main__":
    main()

## DeBERTa_v3

In [ ]:
import argparse
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)


class EmotionDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: np.array(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = int(self.labels[idx])
        return item

    def __len__(self):
        return len(next(iter(self.encodings.values())))


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    acc = (preds == labels).mean()
    return {"macro_f1": macro_f1, "accuracy": acc}


def main():
    # parser = argparse.ArgumentParser(description="Train DeBERTa-v3 for emotion classification.")
    # parser.add_argument("--data_dir", type=str, default="../Processed_data")
    # parser.add_argument("--output_root", type=str, default="../training_output")
    # parser.add_argument("--model_name", type=str, default="microsoft/deberta-v3-base")
    # parser.add_argument("--epochs", type=int, default=3)
    # parser.add_argument("--train_batch_size", type=int, default=32)
    # parser.add_argument("--eval_batch_size", type=int, default=128)
    # parser.add_argument("--max_length", type=int, default=128)
    # parser.add_argument("--tb_logdir", type=str, default="../training/logs/tb/deberta_v3")
    # args = parser.parse_args()

    args = {
        "data_dir": "./data/",
        "output_root": "./",
        "model_name": "microsoft/deberta-v3-base",
        "epochs": 3,
        "train_batch_size": 32,
        "eval_batch_size": 128,
        "max_length": 128,
        "tb_logdir": "./DeBERTa_v3_results/logs/tb/deberta_v3",
    }
    # Convert dict to argparse.Namespace for compatibility
    args = argparse.Namespace(**args)

    data_dir = Path(args.data_dir)
    output_root = Path(args.output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    run_dir = output_root / "deberta_v3"
    run_dir.mkdir(parents=True, exist_ok=True)

    print(f"==== 训练 {args.model_name} ====")
    print(f"数据目录: {data_dir}")
    print(f"输出目录: {run_dir}")

    train_df = pd.read_csv(data_dir / "train.csv")
    valid_df = pd.read_csv(data_dir / "valid.csv")
    test_df = pd.read_csv(data_dir / "test_no_label.csv")

    tokenizer = AutoTokenizer.from_pretrained(args.model_name, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        args.model_name,
        num_labels=7,
    )

    train_enc = tokenizer(
        train_df["text"].tolist(),
        truncation=True,
        padding=True,
        max_length=args.max_length,
    )
    valid_enc = tokenizer(
        valid_df["text"].tolist(),
        truncation=True,
        padding=True,
        max_length=args.max_length,
    )
    test_enc = tokenizer(
        test_df["text"].tolist(),
        truncation=True,
        padding=True,
        max_length=args.max_length,
    )

    train_ds = EmotionDataset(train_enc, train_df["label"].tolist())
    valid_ds = EmotionDataset(valid_enc, valid_df["label"].tolist())
    test_ds = EmotionDataset(test_enc, labels=None)

    training_args = TrainingArguments(
        output_dir=str(run_dir / "checkpoints"),
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.train_batch_size,
        per_device_eval_batch_size=args.eval_batch_size,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=50,
        save_strategy="steps",
        save_steps=2000,
        save_total_limit=5,
        logging_dir=str(Path(args.tb_logdir)),
        report_to=["tensorboard"],
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    valid_outputs = trainer.predict(valid_ds)
    valid_preds = np.argmax(valid_outputs.predictions, axis=-1)
    valid_macro_f1 = f1_score(valid_df["label"].values, valid_preds, average="macro")

    valid_pred_path = Path("deberta_v3_valid_pred.csv")
    pd.DataFrame({"id": valid_df["id"].values, "label": valid_preds}).to_csv(valid_pred_path, index=False)

    test_outputs = trainer.predict(test_ds)
    test_preds = np.argmax(test_outputs.predictions, axis=-1)
    test_pred_path = Path("deberta_v3_test_pred.csv")
    pd.DataFrame({"id": test_df["id"].values, "label": test_preds}).to_csv(test_pred_path, index=False)

    metrics_path = run_dir / "metrics.txt"
    with metrics_path.open("w", encoding="utf-8") as f:
        f.write(f"model={args.model_name}\n")
        f.write(f"valid_macro_f1={valid_macro_f1:.6f}\n")

    print(f"训练完成，验证集 Macro-F1 = {valid_macro_f1:.4f}")
    print(f"验证集预测保存到: {valid_pred_path}")
    print(f"测试集预测保存到: {test_pred_path}")
    print(f"指标保存到: {metrics_path}")
    print(f"TensorBoard 日志目录: {Path(args.tb_logdir)}")


if __name__ == "__main__":
    main()

==== 训练 microsoft/deberta-v3-base ====
数据目录: data
输出目录: deberta_v3


Loading weights: 100%|##########| 198/198 [00:00<00:00, 71716.08it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dens

## distilbert-base-uncased

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score,confusion_matrix
from collections import Counter
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification,Trainer, TrainingArguments
import time
import matplotlib.pyplot as plt
from collections import defaultdict
from datasets import Dataset
import json

# ── Hyper-parameters ──────────────────────────────────────────────────────────
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 7
MAX_LEN     = 64
EPOCHS      = 6
BATCH_SIZE  = 64
LR          = 2e-5
DROPOUT     = 0.3
ATTE_DROPOUT= 0
SEED        = 42
WEIGHT_DECAY= 0
VALIDATION_STEPS = 128

torch.manual_seed(SEED)

def plot_training_curves(history, save_path='training_curves.png'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss
    ax1.plot(history['steps'], history['train_loss'], 'b-', label='Train Loss', alpha=0.7)
    val_steps = [h['step'] for h in history['val_history']]
    val_losses = [h['loss'] for h in history['val_history']]
    ax1.plot(val_steps,val_losses, 'r--', alpha=0.7, label='Validation Loss')
    ax1.scatter(val_steps, val_losses, c='red', s=50, zorder=5)
    ax1.set_xlabel('Training Steps')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # F1
    # if history['train_f1'] != None:
    #     ax2.plot(history['steps'], history['train_f1'], 'g-', label='Train F1', alpha=0.7)
    val_steps = [h['step'] for h in history['val_history']]
    val_f1s = [h['f1'] for h in history['val_history']]
    ax2.plot(val_steps, val_f1s,'--', color='orange', alpha=0.7, label='Validation F1')
    ax2.scatter(val_steps, val_f1s, c='orange', s=50, zorder=5)
    ax2.set_xlabel('Training Steps')
    ax2.set_ylabel('F1 Score')
    ax2.set_title('Training and Validation F1 Score')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=100)
    plt.show()
    print(f"Training curves saved to {save_path}")
    
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    cm = confusion_matrix(labels, preds)
    
    return {
        "accuracy": acc,
        "f1_macro": f1,
        "confusion_matrix": cm.tolist()
    }

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    train_path = "./data/train.csv"
    valid_path = "./data/valid.csv"
    test_path = "./data/test_no_label.csv"
    
    # train_path = "../data/train.csv"
    # valid_path = "../data/valid.csv"
    # test_path = "../data/test_no_label.csv"
    
    train_df = pd.read_csv(train_path)
    valid_df = pd.read_csv(valid_path)
    test_df  = pd.read_csv(test_path)
    train_ds = Dataset.from_pandas(train_df[["text", "label"]])
    valid_ds = Dataset.from_pandas(valid_df[["text", "label"]])
    test_ds  = Dataset.from_pandas(test_df[["id", "text"]])
    
    print("Building dataset …")
    tokenizer=DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        
    print("Tokenizing datasets...")
    train_ds = train_ds.map(tokenize_function, batched=True)
    valid_ds = valid_ds.map(tokenize_function, batched=True)
    test_ds  = test_ds.map(tokenize_function, batched=True)
    
    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs = EPOCHS,
        eval_strategy="steps",
        save_strategy="steps",
        save_steps=VALIDATION_STEPS,
        eval_steps=VALIDATION_STEPS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=512,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        fp16=True,
        seed=SEED,
        logging_dir="./logs",
        logging_steps=VALIDATION_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True
    )

    print("Get model...")
    model=DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=NUM_CLASSES,
        dropout=DROPOUT,
        attention_dropout=ATTE_DROPOUT
    ).to(DEVICE)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        compute_metrics=compute_metrics
    )
    
    print("Training Start...")
    trainer.train()

    history = {
        'steps': [log['step'] for log in trainer.state.log_history if 'loss' in log],
        'train_loss': [log['loss'] for log in trainer.state.log_history if 'loss' in log],
        'train_f1': [log['f1_macro'] for log in trainer.state.log_history if 'f1_macro' in log],
        'val_history': [{'step': log['step'], 'loss': log['eval_loss'], 'f1': log['eval_f1_macro']} 
                        for log in trainer.state.log_history if 'eval_loss' in log]
    }

    print("\nEvaluating on validation set...")
    eval_results = trainer.predict(valid_ds)
    valid_preds = np.argmax(eval_results.predictions, axis=-1)
    
    valid_out = pd.DataFrame({
        "id": valid_df["id"].values,
        "true_label": valid_df["label"].values,
        "label": valid_preds
    })
    valid_out.to_csv("distilbert_valid.csv", index=False)
    print("Saved distilbert_valid.csv")
    
    print("\nGenerating test predictions...")
    test_predictions = trainer.predict(test_ds)
    test_preds = np.argmax(test_predictions.predictions, axis=-1)
    
    out = pd.DataFrame({"id": test_df["id"], "label": test_preds})
    out.to_csv("distilbert_pred.csv", index=False)
    print("Saved test data: distilbert_pred.csv")
    
    with open("history.json", 'w') as f:
        json.dump(history, f, indent=2)
    print("Saved history.json")
    
    if len(history['steps']) > 0:
        plot_training_curves(history, save_path='training_curves.png')
        print("Saved training curves")
    
    print("\n" + "="*50)
    print("Training completed!")
    print(f"Best validation F1-macro: {max([h['f1'] for h in history['val_history']], default=0):.4f}")
    print(f"Validation predictions saved with {len(valid_preds)} samples")
    print("="*50)

if __name__ == "__main__":
    main()

## Qwen3.5-4b LoRA-alpha-16

In [ ]:
# pip install torch transformers accelerate peft bitsandbytes datasets scikit-learn
import os
import torch
from transformers import (
    Qwen3_5Tokenizer,
    AutoTokenizer, 
    Qwen3_5ForSequenceClassification,
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)
from peft import LoraConfig, TaskType, get_peft_model
from datasets import Dataset
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import torch.nn as nn
import tensorboard

epochs = 4
lr = 2e-5
num_labels = 7
gamma = 2
name = './lora-qwen3-4b-l'

train_df = pd.read_csv('./data/train.csv')
valid_df = pd.read_csv('./data/valid.csv')

train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
valid_dataset = Dataset.from_pandas(valid_df[['text', 'label']])

model_name = "Qwen/Qwen3.5-4B"
tokenizer = Qwen3_5Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_valid = valid_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

base_model = Qwen3_5ForSequenceClassification.from_pretrained(
    model_name,
    device_map="auto",  
    # load_in_4bit=True,
    dtype=torch.bfloat16,
    trust_remote_code=True
)

# print(base_model)

# replace (score)
base_model.score = nn.Linear(base_model.score.in_features, num_labels)
base_model.score.requires_grad_()
base_model.config.num_labels = num_labels

# print(base_model)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,       
    lora_alpha=32,  
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], 
    lora_dropout=0.1,
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters() 

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    macro_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    accuracy = accuracy_score(labels, predictions)
    return {"macro_f1": macro_f1, "accuracy": accuracy}

labels_train = train_df['label'].values
class_counts = np.bincount(labels_train, minlength=num_labels)
total = len(labels_train)
alpha = total / (num_labels * (class_counts + 1e-6)) 
alpha = torch.tensor(alpha, dtype=torch.float)

training_args = TrainingArguments(
    output_dir=name,
    dataloader_num_workers=16,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=128,
    learning_rate=lr,              
    num_train_epochs=epochs,

    lr_scheduler_type='linear',
    warmup_steps=100,
    weight_decay=0.01,

    logging_steps=50,

    eval_strategy="steps",
    eval_steps=200,

    save_strategy="no",
    # save_steps=450,
    # save_total_limit=2,

    # load_best_model_at_end=True,
    metric_for_best_model="macro_f1",

    bf16=True,                        

    remove_unused_columns=False,
    report_to=["tensorboard"],                
    logging_dir="./logs",                    
    run_name=name 
)

trainer = Trainer(
# trainer = ImbalanceTrainer(
#     alpha=alpha,
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

output_path = os.path.join(name, "merged")
os.makedirs(output_path, exist_ok=True)
merged_model = model.merge_and_unload()
merged_model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# 自定义数据集类，用于批量编码文本
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),  # 去除 batch 维度
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }

def predict(model, dataloader, device):
    """批量预测函数，返回预测标签列表"""
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Predicting"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            predictions.extend(preds)
    return predictions

def main(model_path : str = 'lora-qwen3-4b-l/merged', res_suffix : str = 'l'):
    # 配置路径
    valid_path = "./data/valid.csv"             # 验证集路径
    test_path = "./data/test_no_label.csv"      # 测试集路径
    valid_out = f"qwen_valid.csv"                  # 验证集输出文件
    test_out = f"qwen_pred.csv"                    # 测试集输出文件

    # 设备设置
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 加载 tokenizer 和模型（合并后的完整模型）
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    ).to(device)

    # ---------- 验证集预测 ----------
    print("Processing validation set...")
    valid_df = pd.read_csv(valid_path)
    valid_texts = valid_df['text'].tolist()

    valid_dataset = TextDataset(valid_texts, tokenizer)
    valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)
    valid_preds = predict(model, valid_loader, device)

    # 将预测结果添加到原 DataFrame 并保存
    valid_df['label'] = valid_preds
    valid_df = valid_df.drop(columns=['text'])
    valid_df.to_csv(valid_out, index=False)
    print(f"Saved {valid_out}")

    # ---------- 测试集预测 ----------
    print("Processing test set...")
    test_df = pd.read_csv(test_path)
    test_texts = test_df['text'].tolist()

    test_dataset = TextDataset(test_texts, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    test_preds = predict(model, test_loader, device)

    # 生成只包含 id 和 label 的输出文件
    test_output = pd.DataFrame({
        'id': test_df['id'],
        'label': test_preds
    })
    test_output.to_csv(test_out, index=False)
    print(f"Saved {test_out}")

if __name__ == "__main__":
    main()

## Voting

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix,accuracy_score,f1_score,recall_score,precision_score
import seaborn as sns
import matplotlib.pyplot as plt 
import numpy as np
from scipy.stats import mode

def load_pred(file_paths):
    all_pred=[]
    
    for file_path in file_paths:
        df=pd.read_csv(file_path)
        all_pred.append(df['label'].values) 
        
    all_pred=np.array(all_pred)
    return all_pred

#voting ------------------------------------
def voting(all_pred,true_labels):
    if len(all_pred.shape)==1:
        vote_pred=all_pred
    else: 
        vote_pred=mode(all_pred,axis=0,keepdims=True)
        vote_pred=vote_pred.mode[0]
    accuracy= accuracy_score(true_labels,vote_pred)
    conf= confusion_matrix(true_labels,vote_pred)
    f1=f1_score(true_labels,vote_pred,average='macro')
    recall = recall_score(true_labels,vote_pred,average='macro')
    precision = precision_score(true_labels,vote_pred,average='macro')
    
    return accuracy,conf,f1, recall, precision

#Drawing----------------------------------
def conf_draw(conf,labels):
    fig,ax=plt.subplots()
    sns.heatmap(
        conf,
        annot=True,
        fmt='d',
        cmap='YlGnBu',
        xticklabels=labels,
        yticklabels=labels,
        ax=ax
    )
    ax.set_xlabel("Predicted Labels")
    ax.set_ylabel("True Labels")
    ax.set_title("Confusion Matrix of Validation")
    
    plt.tight_layout()
    plt.show()
    
# Main
if __name__ =="__main__":
    file_paths=[
        'roberta_valid.csv',
        'deberta_v3_valid_pred.csv',
        'qwen_valid.csv',
        'distilbert_valid.csv'
    ]
    test_paths=[
        'roberta_pred.csv',
        'deberta_v3_test_pred.csv',
        'qwen_pred.csv',
        'distilbert_pred.csv'
    ]
    emotion_labels=['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
    
    true_df=pd.read_csv('./data/valid.csv')
    true_labels=true_df['label'].values

    for file in file_paths:
        print(f"For {file}:")
        f=pd.read_csv(file)
        acc,conf,f1,recall,precision=voting(all_pred=f['label'].values,true_labels=true_labels)
        print(f"Accuracy: {acc:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"F1-score: {f1:.4f}")
        print("="*20)
    
        
    all_pred=load_pred(file_paths=file_paths)
    acc,conf,f1,recall,precision=voting(all_pred=all_pred,true_labels=true_labels)
    print(f"Accuracy: {acc:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"F1-score: {f1:.4f}")
    conf_draw(conf,emotion_labels)
    
    #Save test vote data
    all_test=load_pred(file_paths=test_paths)
    vote_pred=mode(all_test,axis=0,keepdims=True)
    vote_pred=vote_pred.mode[0]
    final_test = pd.DataFrame({
        'id': pd.read_csv(test_paths[0])['id'].copy(),
        'label': vote_pred
    })
    final_test.to_csv('test_pred.csv', index=False)
    print("Saved.")